# ViRL39K · RFT(어려운 문항의 자기 풀이 학습) · Qwen3-VL-4B BF16 LoRA

**이전 RFT(PMC·AQUA)는 baseline과 같았습니다(63.78% 대 63.89%).** 원인은 4번 중 3~4번 맞히는 쉬운 문항만 학습해서 학습 신호가 없었던 것입니다. 이번에는 **모델이 어려워하는 문항(1~3번 정답)** 에서 정답 풀이를 골라 학습합니다.

**데이터:** ViRL39K(TIGER-Lab, 정답이 검증 가능한 문제 38,870건, 카드 라이선스 MIT, 상류 데이터 약관은 제각각)에서 영어·이미지 1장·정답 12자 이하·중복 제거·텍스트 전용(DeepScaleR) 제외 후 카테고리 쿼터로 추출한 약 6,400건 → MMMU validation·test·dev·MMMU-Pro 전체에 대한 누수 검사(pHash≤10, CLIP≥0.97, 텍스트≥0.84)로 397건 격리 → 빈 이미지 제외 **5,996건**. 이 풀은 로컬에서 만들어 `virl39k_pool.zip`(433MB)으로 Drive에 올립니다. 질문·정답·이미지가 들어 있으니 GitHub에 올리지 마세요.

| 카테고리 | 건수 |
|---|---|
| 차트·도표 | 1,509 |
| 비기하 수학 | 1,451 |
| 과학 | 895 |
| 공간 추론 | 620 |
| 기하 | 616 |
| 고급 STEM | 456 |
| 사회과학 | 449 |

**방법**
1. base 4B가 각 문항을 4번 풉니다(평가와 같은 프롬프트·해상도·샘플링, vLLM 0.11.0).
2. **정답 판정:** 객관식은 평가와 같은 v2 파서, 주관식(숫자·수식·단답)은 `\boxed`/`Answer:` 추출 + 숫자 허용오차 + `math-verify`. 잘린 응답은 정답으로 세지 않습니다.
3. **선별:** 주관식은 4번 중 1~3번, 객관식은 2~3번 정답인 문항을 사용합니다(주관식은 찍어서 맞히기 어려워 1번도 신뢰). 4번 모두 맞힌 문항은 전체의 약 12%까지만 넣어 형식을 유지합니다. 정답 풀이 중 **길이가 중앙값에 가까운 것**을 고릅니다(최단 풀이는 "짧게 답하라"를 가르칠 위험, 무작위는 반복 풀이가 섞일 위험).
4. 카테고리 상한으로 수학·기하 쏠림을 막습니다.

**실행 순서와 관문**
1. 설정 → 풀 불러오기 → 프롬프트 검사 → vLLM 환경 → **파일럿(카테고리별 40문항 × 4샘플, 약 280문항)**. 기본값 `PILOT_ONLY=True`라 여기서 멈춥니다.
2. 파일럿 보고서의 **관문 판정**을 확인합니다: 카테고리별로 1~3/4 정답 비율이 15% 이상이고 4/4가 70% 이하여야 통과. 못 넘은 카테고리는 `EXCLUDE_CATEGORIES`에 넣고, 예상 시간·유닛을 보고 `MAX_GEN_QUESTIONS`를 조절한 뒤 `PILOT_ONLY=False`로 파일럿 셀부터 다시 실행합니다.
3. 전체 생성 → 선별(학습 샘플 수 확인) → SFT 학습(adapter 4개) → 병합 → 평가 노트북.

> **런타임:** A100 80GB. **유닛 예산:** 파일럿 약 5, 전체 생성 약 40~60(수학 풀이가 길어 PMC보다 오래 걸림), 학습 약 15~20, 평가 약 25. 누적 130유닛을 넘기면 중단하세요.
>
> **필요한 Drive 파일:** `MyDrive/VIRL39K_RFT_4B/virl39k_pool.zip` (로컬 `mutimodal_private/virl39k/virl39k_pool.zip`), 평가 프로젝트 `MyDrive/MMDL/code/qwen3_vl_mmmu_eval`.

In [ ]:
%pip -q install "transformers==4.57.1" "trl==0.24.0" "peft==0.17.1" "accelerate==1.11.0" "datasets==4.3.0" "huggingface_hub==0.36.0" "pandas>=2" "scikit-learn>=1.4" "pillow>=10" "tqdm>=4.66" "pyyaml>=6" "math-verify>=0.5"

## 1. 설정 · 평가 조건 · revision 고정
평가 프로젝트의 `final_16k.yaml`에서 해상도와 샘플링 값을 읽고 `prompt.py`·`parse_v2.py`를 import합니다. 학습 데이터 생성과 평가가 같은 코드를 씁니다.

In [ ]:
import os, re, io, gc, json, math, random, hashlib, zipfile, shutil, time, subprocess, sys
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np, pandas as pd, torch, yaml
from PIL import Image, ImageOps
from tqdm.auto import tqdm
from huggingface_hub import HfApi

SEED=3407
MODEL_ID='Qwen/Qwen3-VL-4B-Instruct'; MODEL_REV='ebb281ec70b05090aa6165b016eac8ec08e71b17'   # baseline과 동일

from google.colab import drive
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/VIRL39K_RFT_4B'); ROOT.mkdir(parents=True,exist_ok=True)
POOL_ZIP=ROOT/'virl39k_pool.zip'     # 로컬에서 만든 풀(누수 검사 통과분)을 여기에 업로드
WORK=Path('/content/rft_work'); WORK.mkdir(parents=True,exist_ok=True)
POOL_DIR=WORK/'pool'
DRIVE_EVAL_PROJECT=Path('/content/drive/MyDrive/MMDL/code/qwen3_vl_mmmu_eval')

# ===== 생성 =====
RFT_SAMPLES=4
RFT_MIN_CORRECT=1         # (기록용) 주관식 최소 정답 횟수. 실제 규칙은 HARD_RULES
GEN_MAX_TOKENS=6144        # 수학 풀이가 길어서 PMC(4096)보다 큼. 이보다 길면 잘린 것으로 보고 정답으로 세지 않음
GEN_MAX_MODEL_LEN=14336    # 이미지 최대 5,120토큰 + 프롬프트 + 출력 6,144
GEN_GPU_UTIL=0.85; GEN_CHUNK_QUESTIONS=256
PILOT_PER_CATEGORY=40
PILOT_ONLY=True            # True: 파일럿 보고서 저장 후 멈춤. 확인 후 False로 바꿔 파일럿 셀부터 재실행
EXCLUDE_CATEGORIES=[]      # 파일럿 관문에서 탈락한 카테고리를 여기에 넣음
MAX_GEN_QUESTIONS=0        # 0=제한 없음. 예산이 부족하면 카테고리 비율대로 줄임 (예: 4500)
VAL_FRACTION=0.05
# ===== 선별 =====
HARD_RULES={'open':(1,3),'mc':(2,3)}   # 종류별 (최소,최대) 정답 횟수: 어려운 문항
ANCHOR_FRACTION=0.12                   # 4/4 정답 문항을 전체 학습 샘플의 이 비율까지 추가(형식 유지)
CATEGORY_CAP={'(GradeSchool) Geometric':350,'(GradeSchool) Non-Geo Math':800}   # 쏠림 방지(어려운 문항 기준 상한)
MAX_TRAIN_SAMPLES=2600
MIN_TARGET_TOKENS=8; MAX_TARGET_TOKENS=4096
TARGET_PICK='median_length'            # 정답 풀이 중 길이가 중앙값에 가장 가까운 것
GATE_MIN_HARD_SHARE=0.15; GATE_MAX_EASY_SHARE=0.70
MIN_TRAIN_SAMPLES=800
# ===== 학습 =====
LEARNING_RATE=2e-5; NUM_TRAIN_EPOCHS=2.0; GRADIENT_ACCUMULATION=8
ADAPTER_SAVE_PCTS=(25,50,75,100); SAVE_STEPS=250
RUN_NAME='virl39k_rft4b_bf16_lora_4adapters'
AUTO_SHUTDOWN=True

assert torch.cuda.is_available(),'GPU 런타임(A100)을 선택하세요.'
gpu_name=torch.cuda.get_device_name(0); gpu_gib=torch.cuda.get_device_properties(0).total_memory/2**30
print('GPU',gpu_name,f'{gpu_gib:.1f} GiB')
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)

# --- 평가 프로젝트: 프롬프트·파서·생성 설정을 그대로 사용 ---
cands=[DRIVE_EVAL_PROJECT,DRIVE_EVAL_PROJECT/'code'/'qwen3_vl_mmmu_eval']
PROJECT=next((p for p in cands if (p/'configs'/'final_16k.yaml').is_file() and (p/'src'/'mmmu_eval'/'parse_v2.py').is_file()),None)
assert PROJECT is not None,f'평가 프로젝트를 찾지 못했습니다: {cands}'
if str(PROJECT/'src') not in sys.path: sys.path.insert(0,str(PROJECT/'src'))
from mmmu_eval.prompt import render_text, DEFAULT_PROMPT
from mmmu_eval.parse_v2 import extract_answer_v2
EVAL_CFG=yaml.safe_load((PROJECT/'configs'/'final_16k.yaml').read_text(encoding='utf-8'))
EXPECTED={'temperature':0.7,'top_p':0.8,'top_k':20,'repetition_penalty':1.0,'presence_penalty':1.5,
          'min_pixels':1310720,'max_pixels':5242880,'seed':3407}
for k,v in EXPECTED.items():
    assert EVAL_CFG[k]==v,f'final_16k.yaml {k}={EVAL_CFG[k]} (baseline {v})'
assert EVAL_CFG['prompt']==DEFAULT_PROMPT, 'final_16k.yaml prompt가 prompt.py 기본값과 다릅니다'
PARSER_REGEX=EVAL_CFG['parser']['regex']
MIN_PIXELS=EVAL_CFG['min_pixels']; MAX_PIXELS=EVAL_CFG['max_pixels']
print('평가 프로젝트:',PROJECT)
print('생성/학습 해상도:',MIN_PIXELS,MAX_PIXELS,'| 샘플링:',{k:EVAL_CFG[k] for k in ('temperature','top_p','top_k','presence_penalty','repetition_penalty')})


api=HfApi()
revs={'model':api.repo_info(repo_id=MODEL_ID,repo_type='model',revision=MODEL_REV).sha,
      'virl39k':'TIGER-Lab/ViRL39K@812ec617dea4bc8a4e751663b88e4ebb7de4d00e'}
assert revs['model']==MODEL_REV
import importlib.metadata as ilm
versions={p:ilm.version(p) for p in ['transformers','trl','peft','datasets','accelerate','torch','math-verify']}
versions.update(gpu=gpu_name,seed=SEED)
(ROOT/'environment.json').write_text(json.dumps(versions,indent=2))
def jdump(o):
    return json.dumps(o,indent=2,ensure_ascii=False,default=lambda x:x.item() if hasattr(x,'item') else str(x))
print('Revision:',json.dumps(revs,indent=2))

## 2. 풀 불러오기 · 프롬프트 = 평가 프롬프트 검사 · train/val 분리
- `virl39k_pool.zip`을 풀고 해시를 확인합니다. 모든 문항의 `prompt_text`가 평가 코드의 `render_text`와 같은지 검사합니다(객관식은 `Question/Options/A.…/Please select…`, 주관식은 `Question: …`).
- val은 학습에 쓰지 않고 학습 후 loss 확인에만 씁니다(카테고리별 5%).

In [ ]:
assert POOL_ZIP.is_file(),f'풀 파일이 없습니다: {POOL_ZIP}. 로컬의 virl39k_pool.zip을 Drive에 올리세요.'
if not (POOL_DIR/'candidates.jsonl').is_file():
    shutil.rmtree(POOL_DIR,ignore_errors=True); POOL_DIR.mkdir(parents=True)
    with zipfile.ZipFile(POOL_ZIP) as z: z.extractall(POOL_DIR)
pool_man=json.loads((POOL_DIR/'pool_manifest.json').read_text())
assert hashlib.sha256((POOL_DIR/'candidates.jsonl').read_bytes()).hexdigest()==pool_man['candidates_sha256'],'풀 파일 해시 불일치'
cands=pd.DataFrame([json.loads(l) for l in (POOL_DIR/'candidates.jsonl').open(encoding='utf-8') if l.strip()])
DATA_SIG=hashlib.sha256(json.dumps({'pool':pool_man['candidates_sha256'],'seed':SEED,'val_fraction':VAL_FRACTION},sort_keys=True).encode()).hexdigest()[:16]
assert len(cands)==pool_man['rows'] and cands.uid.is_unique
cands['image']=[str(POOL_DIR/p) for p in cands.image]
bad=[]
for p in tqdm(cands.image,desc='verify images'):
    try:
        with Image.open(p) as im: im.verify()
    except Exception as e: bad.append((p,repr(e)))
assert not bad,f'열리지 않는 이미지 {len(bad)}개: {bad[:3]}'
def student_text(r):
    return render_text(r['question'],list(r['choices']),EVAL_CFG['prompt'])
mism=[r['uid'] for r in cands.to_dict('records') if student_text(r)!=r['prompt_text']]
assert not mism,f'프롬프트가 평가 프롬프트와 다른 문항 {len(mism)}개: {mism[:3]}'
assert all(len(c)<=9 for c in cands.choices) and cands[cands.kind=='mc'].gold.isin(list('ABCDEFGHI')).all()
# train/val 분리: 카테고리별 5%
rng=np.random.RandomState(SEED); cands['split']='train'
for cat,idx in cands.groupby('category').groups.items():
    idx=list(idx); val_idx=rng.choice(idx,size=max(1,int(len(idx)*VAL_FRACTION)),replace=False); cands.loc[val_idx,'split']='val'
print('후보:',len(cands),'| split',cands.split.value_counts().to_dict())
print(cands.groupby(['category','kind']).size().unstack(fill_value=0).to_string())
ex=cands[cands.kind=='open'].iloc[0]; print('\n[주관식 프롬프트 예]',repr(student_text(ex))); ex=cands[cands.kind=='mc'].iloc[0]; print('[객관식 프롬프트 예]',repr(student_text(ex)))

## 3. vLLM 생성 환경 (별도 Python 3.12 · 평가와 같은 버전)
RFT 노트북과 같은 방식입니다. 문항마다 seed를 `3407+j`(j=0~3)로 고정해 4번 생성하고, 256문항마다 Drive에 저장하며 다시 실행하면 이어서 생성합니다.

In [ ]:
GEN_VENV=Path('/content/rft_vllm_venv'); GEN_PY=GEN_VENV/'bin'/'python'
GEN_PACKAGES=['vllm==0.11.0','transformers==4.57.1','qwen-vl-utils==0.0.14','pillow==11.3.0']
def run_checked(cmd):
    r=subprocess.run([str(c) for c in cmd],capture_output=True,text=True)
    if r.returncode:
        print(r.stdout[-3000:]);print(r.stderr[-3000:])
        raise RuntimeError(f'명령 실패 (exit {r.returncode}): {" ".join(map(str,cmd))}')
    return r
marker=GEN_VENV/'installed_spec.json'
if not (marker.exists() and json.loads(marker.read_text())==GEN_PACKAGES):
    shutil.rmtree(GEN_VENV,ignore_errors=True)
    run_checked([sys.executable,'-m','pip','install','-q','uv'])
    run_checked([sys.executable,'-m','uv','venv','--python','3.12',str(GEN_VENV)])
    print('vLLM 환경 설치 중 (수 분)...')
    run_checked([sys.executable,'-m','uv','pip','install','--python',str(GEN_PY),*GEN_PACKAGES])
    marker.write_text(json.dumps(GEN_PACKAGES))
print(run_checked([GEN_PY,'-c','import sys,torch,vllm,transformers;print(sys.version.split()[0],torch.__version__,vllm.__version__,transformers.__version__)']).stdout.strip())

GEN_SCRIPT=WORK/'rft_generate.py'
GEN_SCRIPT.write_text(r"""
import argparse, json, os, time
from pathlib import Path


def main():
    ap = argparse.ArgumentParser()
    for name in ("requests", "out", "model", "revision", "timing"):
        ap.add_argument("--" + name, required=True)
    for name in ("samples", "max-tokens", "min-pixels", "max-pixels", "chunk", "max-model-len", "seed", "top-k"):
        ap.add_argument("--" + name, type=int, required=True)
    for name in ("temperature", "top-p", "presence-penalty", "repetition-penalty", "gpu-util"):
        ap.add_argument("--" + name, type=float, required=True)
    a = ap.parse_args()
    os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")
    reqs = [json.loads(l) for l in open(a.requests, encoding="utf-8") if l.strip()]
    out = Path(a.out)
    done = set()
    if out.exists():
        for l in out.open(encoding="utf-8"):
            try:
                rec = json.loads(l)
                if len(rec["samples"]) == a.samples:
                    done.add(rec["uid"])
            except Exception:
                pass
        data = out.read_bytes()
        if data and not data.endswith(b"\n"):      # torn last line from an interrupted run
            with out.open("ab") as f:
                f.write(b"\n")
    todo = [r for r in reqs if r["uid"] not in done]
    print(f"requests {len(reqs)} done {len(reqs) - len(todo)} todo {len(todo)}", flush=True)
    if not todo:
        return
    t0 = time.monotonic()
    from transformers import AutoProcessor
    from qwen_vl_utils import process_vision_info
    from vllm import LLM, SamplingParams
    processor = AutoProcessor.from_pretrained(a.model, revision=a.revision)
    # 평가 코드(src/mmmu_eval/generate.py)와 같은 엔진 설정. generation_config="vllm": 모델 폴더의 기본 샘플링 값을 쓰지 않음
    llm = LLM(model=a.model, revision=a.revision, dtype="bfloat16", max_model_len=a.max_model_len,
              gpu_memory_utilization=a.gpu_util, limit_mm_per_prompt={"image": 1}, seed=a.seed,
              generation_config="vllm")
    load_seconds = time.monotonic() - t0

    def params(j):
        # 문항당 RFT 샘플 j는 seed+j로 고정. 평가와 같은 샘플링 값(temperature/top_p/top_k/penalty)
        return SamplingParams(temperature=a.temperature, top_p=a.top_p, top_k=a.top_k,
                              repetition_penalty=a.repetition_penalty, presence_penalty=a.presence_penalty,
                              seed=a.seed + j, max_tokens=a.max_tokens, n=1, stop_token_ids=[])

    def prep(r):
        # src/mmmu_eval/prompt.py build_messages + generate.py prepare_inputs_for_vllm과 같은 처리
        msgs = [{"role": "user", "content": [
            {"type": "image", "image": r["image"], "min_pixels": a.min_pixels, "max_pixels": a.max_pixels},
            {"type": "text", "text": r["text"]}]}]
        text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        images, videos, kw = process_vision_info(msgs, image_patch_size=processor.image_processor.patch_size,
                                                 return_video_kwargs=True, return_video_metadata=True)
        return {"prompt": text, "multi_modal_data": {"image": images}, "mm_processor_kwargs": kw}

    gen_seconds = 0.0
    n_done = 0
    with out.open("a", encoding="utf-8") as f:
        for s in range(0, len(todo), a.chunk):
            part = todo[s:s + a.chunk]
            t = time.monotonic()
            prompts = [prep(r) for r in part]
            inputs = [p for p in prompts for _ in range(a.samples)]
            sps = [params(j) for _ in part for j in range(a.samples)]
            outs = llm.generate(inputs, sampling_params=sps, use_tqdm=False)
            assert len(outs) == len(inputs), (len(outs), len(inputs))
            for i, r in enumerate(part):
                group = outs[i * a.samples:(i + 1) * a.samples]
                samples = []
                for j, o in enumerate(group):
                    c = o.outputs[0]
                    samples.append({"text": c.text, "finish_reason": c.finish_reason,
                                    "output_tokens": len(c.token_ids), "seed": a.seed + j})
                f.write(json.dumps({"uid": r["uid"], "input_tokens": len(group[0].prompt_token_ids),
                                    "samples": samples}, ensure_ascii=False) + "\n")
            f.flush()
            os.fsync(f.fileno())
            dt = time.monotonic() - t
            gen_seconds += dt
            n_done += len(part)
            rate = n_done / max(gen_seconds, 1e-9)
            print(f"chunk {s // a.chunk + 1}: {len(part)} questions x {a.samples} in {dt:.0f}s | "
                  f"{n_done}/{len(todo)} | {rate:.2f} q/s | ETA {(len(todo) - n_done) / max(rate, 1e-9) / 60:.0f} min",
                  flush=True)
    with open(a.timing, "a", encoding="utf-8") as tf:
        tf.write(json.dumps({"unix_time": time.time(), "questions": n_done, "samples_per_question": a.samples,
                             "generation_seconds": gen_seconds, "load_seconds": load_seconds}) + "\n")


if __name__ == "__main__":
    main()
""")
run_checked([GEN_PY,'-m','py_compile',GEN_SCRIPT])
print('생성 스크립트:',GEN_SCRIPT)

## 4. 채점·선별 함수
- **객관식:** 평가와 같은 `extract_answer_v2`.
- **주관식:** `\boxed{}` → `Answer:`/`Final Answer:` 줄 → `the answer is` 순으로 최종 답을 뽑고(없으면 오답), gold와 비교합니다. 비교는 정규화 문자열 일치 → 숫자 허용오차(상대 1e-3) → 좌표·구간 비교 → `math-verify` 순서입니다. 무작위로 짝지은 서로 다른 gold 6,000쌍에서 오탐은 10건(0.17%)이었습니다.
- 잘린 응답(`finish_reason≠stop`)과 한중일 문자가 섞인 응답은 학습 타겟 후보에서 제외합니다.

In [ ]:
"""RFT 정답 판정(ViRL39K 풀). 객관식은 평가와 같은 v2 파서(노트북에서 주입), 주관식은 아래 규칙.
이 파일의 내용은 Colab 노트북 셀에 그대로 복사된다(노트북 독립 실행 가능하도록). 로컬 테스트: test_open_answer_scoring.py."""
import re, math, unicodedata

_UNIT = r'(?:square\s+)?(?:cm|mm|km|m|kg|g|mg|s|sec|seconds?|min|minutes?|hours?|hrs?|h|days?|years?|units?|degrees?|meters?|metres?|centimeters?|cm\^?2|cm\^?3|m\^?2|m\^?3|n|j|w|v|a|hz|pa|mol|l|ml|yuan|dollars?|cents?|pieces?|people|persons?|times?)'

def _strip_math(s):
    s = str(s)
    s = re.sub(r'\\text\s*\{([^{}]*)\}', r'\1', s); s = re.sub(r'\\mathrm\s*\{([^{}]*)\}', r'\1', s)
    s = s.replace('\\dfrac', '\\frac').replace('\\tfrac', '\\frac').replace('\\left', '').replace('\\right', '')
    s = s.replace('\\,', '').replace('\\!', '').replace('\\ ', ' ').replace('\\%', '%').replace('\\$', '')
    s = s.replace('^{\\circ}', '°').replace('^\\circ', '°').replace('\\circ', '°').replace('$', '')
    s = s.replace('π', '\\pi').replace('√', '\\sqrt')
    s = unicodedata.normalize('NFKC', s)
    return s.strip()

def _norm(s):
    s = _strip_math(s).casefold().strip(' .,;:!?*"\'')
    s = re.sub(r'^[a-z]\s*=\s*', '', s)                      # "x = 5" -> "5"
    if s.startswith('{') and s.endswith('}') and '{' not in s[1:-1]: s = s[1:-1]
    if s.startswith('(') and s.endswith(')') and ',' not in s:          # (4n+2) -> 4n+2 (좌표·구간은 쉼표가 있어 제외)
        depth = 0
        for k, ch in enumerate(s):
            depth += (ch == '(') - (ch == ')')
            if depth == 0 and k < len(s) - 1: break
        else: s = s[1:-1]
    return re.sub(r'\s+', '', s)

def _strip_units(s):
    s = _norm(s)
    s = re.sub(r'(?<=[\d\}])\s*' + _UNIT + r'$', '', s)
    s = s.replace('°', '').replace('degrees', '').replace('degree', '').replace('%', '')
    s = s.replace('(', '(').strip()
    return s

def _to_float(s):
    t = _strip_units(s).replace(',', '') if re.fullmatch(r'[\d,]+(\.\d+)?', _strip_units(s)) else _strip_units(s)
    t = re.sub(r'\\frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}', r'(\1)/(\2)', t)
    t = re.sub(r'\\sqrt\s*\{([^{}]+)\}', r'sqrt(\1)', t); t = re.sub(r'\\sqrt\s*(\d+)', r'sqrt(\1)', t)
    t = t.replace('\\pi', '*pi').replace('\\cdot', '*').replace('\\times', '*').replace('^', '**')
    t = re.sub(r'(\d)(\(|sqrt|pi)', r'\1*\2', t); t = re.sub(r'\)(\d|\(|sqrt|pi)', r')*\1', t)
    if not t or not re.fullmatch(r'[\d\.\+\-\*/\(\)sqrtpi]+', t) or t.count('(') != t.count(')'): return None
    try:
        v = eval(t, {'__builtins__': {}}, {'sqrt': math.sqrt, 'pi': math.pi})   # 문자 집합을 위 정규식으로 제한
        return float(v) if isinstance(v, (int, float)) and math.isfinite(v) else None
    except Exception:
        return None

def _boxed(text):
    out = []
    for m in re.finditer(r'\\boxed\s*\{', text):
        i, depth = m.end(), 1
        while i < len(text) and depth:
            depth += (text[i] == '{') - (text[i] == '}'); i += 1
        if depth == 0: out.append(text[m.end():i - 1])
    return out[-1].strip() if out else None

def extract_open(response):
    """최종 답 후보: \\boxed 마지막 > 'Answer:' 마지막 줄 > 'the answer is'. 없으면 None."""
    b = _boxed(response)
    if b: return b
    ms = list(re.finditer(r'(?:final\s+answer|answer)\s*(?:is\s*)?[:：]\s*\**\s*(.+)', response, flags=re.I))
    if ms:
        t = ms[-1].group(1).strip().strip('*').strip()
        t = re.sub(r'\s*\*+\s*$', '', t)
        return t.split('\n')[0].strip() if t else None
    ms = list(re.finditer(r'the\s+(?:final\s+)?answer\s+is\s*[:：]?\s*\**\s*([^\n]+?)\s*\**\s*(?:\.\s|\.$|$)', response, flags=re.I | re.M))
    if ms: return ms[-1].group(1).strip().strip('*').strip()
    return None

def _verify_symbolic(pred, gold):
    try:
        from math_verify import parse, verify
        return bool(verify(parse('$' + _strip_math(gold) + '$'), parse('$' + _strip_math(pred) + '$')))
    except Exception:
        return False

def match_open(pred, gold):
    if pred is None: return False
    pn, gn = _norm(pred), _norm(gold)
    if not pn or not gn: return False
    if pn == gn: return True
    pu, gu = _strip_units(pred), _strip_units(gold)
    if pu == gu: return True
    pf, gf = _to_float(pred), _to_float(gold)
    if pf is not None and gf is not None:
        return abs(pf - gf) <= max(1e-6, 1e-3 * abs(gf))
    if re.fullmatch(r'[\(\[].*[\)\]]', gu) or re.fullmatch(r'[\(\[].*[\)\]]', pu):      # 좌표·구간: 공백만 무시
        return re.sub(r'\s+', '', pu) == re.sub(r'\s+', '', gu)
    return _verify_symbolic(pred, gold)


CJK=re.compile(r'[぀-ヿ㐀-䶿一-鿿가-힯]')
GEN_FILE=ROOT/'virl_generations_PRIVATE.jsonl'; TIMING_FILE=ROOT/'rft_generation_timing.jsonl'
LOG_FILE=ROOT/'rft_vllm.log'

def load_generations():
    out={}
    if GEN_FILE.exists():
        for line in GEN_FILE.open(encoding='utf-8'):
            try: rec=json.loads(line)
            except json.JSONDecodeError: continue      # 끊긴 마지막 줄
            if len(rec.get('samples',[]))==RFT_SAMPLES: out[rec['uid']]=rec
    return out

def run_generation(frame,tag):
    req=WORK/f'rft_requests_{tag}.jsonl'
    with req.open('w',encoding='utf-8') as f:
        for r in frame.to_dict('records'):
            f.write(json.dumps({'uid':r['uid'],'image':r['image'],'text':student_text(r)},ensure_ascii=False)+'\n')
    gc.collect();torch.cuda.empty_cache()
    cmd=[GEN_PY,GEN_SCRIPT,'--requests',req,'--out',GEN_FILE,'--model',MODEL_ID,'--revision',revs['model'],
         '--timing',TIMING_FILE,'--samples',RFT_SAMPLES,'--max-tokens',GEN_MAX_TOKENS,
         '--min-pixels',MIN_PIXELS,'--max-pixels',MAX_PIXELS,'--chunk',GEN_CHUNK_QUESTIONS,
         '--max-model-len',GEN_MAX_MODEL_LEN,'--seed',EVAL_CFG['seed'],'--top-k',EVAL_CFG['top_k'],
         '--temperature',EVAL_CFG['temperature'],'--top-p',EVAL_CFG['top_p'],
         '--presence-penalty',EVAL_CFG['presence_penalty'],'--repetition-penalty',EVAL_CFG['repetition_penalty'],
         '--gpu-util',GEN_GPU_UTIL]
    env=os.environ.copy(); env['VLLM_WORKER_MULTIPROC_METHOD']='spawn'
    t0=time.time()
    with open(LOG_FILE,'a',encoding='utf-8') as log:
        log.write(f'\n===== {tag} {time.ctime()} =====\n')
        p=subprocess.Popen([str(c) for c in cmd],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
        for line in p.stdout:
            log.write(line)
            if line.startswith(('chunk','requests')) or 'Error' in line or 'error:' in line.lower(): print(line,end='',flush=True)
        rc=p.wait()
    if rc: raise RuntimeError(f'vLLM 생성 실패 (exit {rc}). 로그: {LOG_FILE}')
    print(f'{tag}: {(time.time()-t0)/60:.1f}분 (모델 로딩 포함)')


def score_question(r,rec):
    rows=[]
    for s in rec['samples']:
        text=s['text']; trunc=s['finish_reason']!='stop'
        if r['kind']=='mc':
            p=extract_answer_v2(text,dict(zip('ABCDEFGHI',r['choices'])),PARSER_REGEX,'multiple-choice',trunc)
            ok=(not trunc) and p['method'] not in ('unparsed','refused') and p['extracted']==r['gold']
            method=p['method']
        else:
            pred=None if trunc else extract_open(text)
            ok=(not trunc) and match_open(pred,r['gold'])
            method='open_ok' if ok else ('open_unparsed' if pred is None else 'open_wrong')
        rows.append({'ok':bool(ok),'truncated':trunc,'tokens':s['output_tokens'],'method':method,'text':text,'cjk':bool(CJK.search(text))})
    return rows

def pick_target(r,sc):
    good=[x for x in sc if x['ok'] and not x['cjk'] and MIN_TARGET_TOKENS<=x['tokens']<=MAX_TARGET_TOKENS]
    if not good: return None
    med=float(np.median([x['tokens'] for x in good]))
    best=min(abs(x['tokens']-med) for x in good)
    close=[x for x in good if abs(x['tokens']-med)==best]
    return random.Random(int(hashlib.sha256(f"{SEED}|target|{r['uid']}".encode()).hexdigest()[:16],16)).choice(close)

def build_quality(frame,gens):
    out=[]
    for r in frame.to_dict('records'):
        rec=gens.get(r['uid'])
        if rec is None: continue
        sc=score_question(r,rec); k=sum(x['ok'] for x in sc); lo,hi=HARD_RULES[r['kind']]
        best=pick_target(r,sc)
        status=('hard' if lo<=k<=hi else 'easy_4of4' if k==RFT_SAMPLES else f'correct_{k}_of_{RFT_SAMPLES}')
        if status in ('hard','easy_4of4') and best is None: status+='_no_valid_target'
        out.append({**{c:r[c] for c in ('uid','source','kind','category','split','group','question','gold','image')},
                    'choices':r['choices'],'n_correct':k,'sample_acc':k/RFT_SAMPLES,'n_truncated':sum(x['truncated'] for x in sc),
                    'tokens_all':[x['tokens'] for x in sc],'tokens_correct':[x['tokens'] for x in sc if x['ok']],
                    'status':status,'target':best['text'] if best else None,'target_tokens':best['tokens'] if best else None})
    return pd.DataFrame(out)

def category_report(q):
    rep={}
    for cat,g in q.groupby('category'):
        toks=pd.Series([t for ts in g.tokens_all for t in ts])
        nc=g.n_correct
        rep[cat]={'questions':len(g),'sample_accuracy':round(float(g.sample_acc.mean()),4),
            'n_correct_dist':{int(k):int(v) for k,v in nc.value_counts().sort_index().items()},
            'hard_share':round(float(g.status.str.startswith('hard').mean()),4),
            'easy_4of4_share':round(float((nc==RFT_SAMPLES).mean()),4),'zero_share':round(float((nc==0).mean()),4),
            'truncated_sample_rate':round(float(g.n_truncated.sum()/(len(g)*RFT_SAMPLES)),4),
            'tokens_median':float(toks.median()),'tokens_p90':float(toks.quantile(0.9)),
            'open_share':round(float((g.kind=='open').mean()),3)}
        rep[cat]['gate_pass']=bool(rep[cat]['hard_share']>=GATE_MIN_HARD_SHARE and rep[cat]['easy_4of4_share']<=GATE_MAX_EASY_SHARE)
    return rep

In [ ]:
# === 패치 v2: 주관식 최종 답 추출 보강 + open_unparsed_share 진단 (채점·선별 함수 셀 바로 뒤, 파일럿 셀 앞에 이 셀을 추가해 실행) ===
def _clean_cand(t):
    t = str(t).strip()
    t = re.sub(r'^[>\s]+', '', t); t = t.replace('✅', '').replace('\\[', '').replace('\\]', '').strip()
    t = re.sub(r'^\*+|\*+$', '', t.strip()).strip()
    return t

def extract_open(response):
    """명시적 표지가 있는 최종 답 후보: \\boxed 마지막 > 'Answer:' 마지막 줄 > 'the answer is'. 없으면 None."""
    b = _boxed(response)
    if b: return b
    ms = list(re.finditer(r'(?:final\s+answer|answer)\s*(?:is\s*)?[:：]\s*\**\s*(.+)', response, flags=re.I))
    if ms:
        t = ms[-1].group(1).strip().strip('*').strip()
        t = re.sub(r'\s*\*+\s*$', '', t)
        return t.split('\n')[0].strip() if t else None
    ms = list(re.finditer(r'the\s+(?:final\s+)?answer\s+is\s*[:：]?\s*\**\s*([^\n]+?)\s*\**\s*(?:\.\s|\.$|$)', response, flags=re.I | re.M))
    if ms: return ms[-1].group(1).strip().strip('*').strip()
    return None

def open_candidates(response):
    """채점에 쓸 후보 목록. 표지가 있으면 그 답(과 그 안의 굵은 조각들)만, 없으면 끝부분의 굵은 글씨 / 마지막 짧은 줄을 후보로 쓴다."""
    ex = extract_open(response)
    if ex:
        c = [_clean_cand(ex)]
        parts = [p for p in re.split(r'\*\*', ex) if p.strip() and p.strip().lower() not in ('or', 'and', ',', '=')]
        c += [_clean_cand(p) for p in parts if len(parts) > 1]
        return [x for x in dict.fromkeys(c) if x]
    tail = response[-400:]
    bold = re.findall(r'\*\*([^*\n]{1,80})\*\*', tail)
    cand = [_clean_cand(bold[-1])] if bold else []
    lines = [l.strip() for l in response.strip().split('\n') if l.strip()]
    if lines:
        last = _clean_cand(lines[-1]).strip('. ')
        if 0 < len(last) <= 40 and re.search(r'[\d\w]', last): cand.append(last)
    return [x for x in dict.fromkeys(cand) if x]

def match_response(response, gold):
    """응답에서 후보 답들을 뽑아 하나라도 gold와 맞으면 정답. (후보 없음 -> (False, 'open_unparsed'))"""
    cands = open_candidates(response)
    if not cands: return False, 'open_unparsed'
    return any(match_open(c, gold) for c in cands), 'open_checked'


def score_question(r,rec):
    rows=[]
    for s in rec['samples']:
        text=s['text']; trunc=s['finish_reason']!='stop'
        if r['kind']=='mc':
            p=extract_answer_v2(text,dict(zip('ABCDEFGHI',r['choices'])),PARSER_REGEX,'multiple-choice',trunc)
            ok=(not trunc) and p['method'] not in ('unparsed','refused') and p['extracted']==r['gold']
            method=p['method']
        else:
            if trunc: ok,method=False,'open_truncated'
            else:
                ok,method=match_response(text,r['gold'])
                method='open_ok' if ok else ('open_unparsed' if method=='open_unparsed' else 'open_wrong')
        rows.append({'ok':bool(ok),'truncated':trunc,'tokens':s['output_tokens'],'method':method,'text':text,'cjk':bool(CJK.search(text))})
    return rows

def pick_target(r,sc):
    good=[x for x in sc if x['ok'] and not x['cjk'] and MIN_TARGET_TOKENS<=x['tokens']<=MAX_TARGET_TOKENS]
    if not good: return None
    med=float(np.median([x['tokens'] for x in good]))
    best=min(abs(x['tokens']-med) for x in good)
    close=[x for x in good if abs(x['tokens']-med)==best]
    return random.Random(int(hashlib.sha256(f"{SEED}|target|{r['uid']}".encode()).hexdigest()[:16],16)).choice(close)

def build_quality(frame,gens):
    out=[]
    for r in frame.to_dict('records'):
        rec=gens.get(r['uid'])
        if rec is None: continue
        sc=score_question(r,rec); k=sum(x['ok'] for x in sc); lo,hi=HARD_RULES[r['kind']]
        best=pick_target(r,sc)
        status=('hard' if lo<=k<=hi else 'easy_4of4' if k==RFT_SAMPLES else f'correct_{k}_of_{RFT_SAMPLES}')
        if status in ('hard','easy_4of4') and best is None: status+='_no_valid_target'
        out.append({**{c:r[c] for c in ('uid','source','kind','category','split','group','question','gold','image')},
                    'choices':r['choices'],'n_correct':k,'sample_acc':k/RFT_SAMPLES,'n_truncated':sum(x['truncated'] for x in sc),
                    'tokens_all':[x['tokens'] for x in sc],'tokens_correct':[x['tokens'] for x in sc if x['ok']],
                    'methods':[x['method'] for x in sc],
                    'status':status,'target':best['text'] if best else None,'target_tokens':best['tokens'] if best else None})
    return pd.DataFrame(out)

def category_report(q):
    rep={}
    for cat,g in q.groupby('category'):
        toks=pd.Series([t for ts in g.tokens_all for t in ts])
        nc=g.n_correct
        rep[cat]={'questions':len(g),'sample_accuracy':round(float(g.sample_acc.mean()),4),
            'n_correct_dist':{int(k):int(v) for k,v in nc.value_counts().sort_index().items()},
            'hard_share':round(float(g.status.str.startswith('hard').mean()),4),
            'easy_4of4_share':round(float((nc==RFT_SAMPLES).mean()),4),'zero_share':round(float((nc==0).mean()),4),
            'truncated_sample_rate':round(float(g.n_truncated.sum()/(len(g)*RFT_SAMPLES)),4),
            'tokens_median':float(toks.median()),'tokens_p90':float(toks.quantile(0.9)),
            'open_share':round(float((g.kind=='open').mean()),3)}
        om=[m for ms,k in zip(g.methods,g.kind) if k=='open' for m in ms if m!='open_truncated']
        rep[cat]['open_unparsed_share']=round(om.count('open_unparsed')/max(1,len(om)),4)   # 답을 못 뽑은 비율(추출 실패 점검용)
        rep[cat]['gate_pass']=bool(rep[cat]['hard_share']>=GATE_MIN_HARD_SHARE and rep[cat]['easy_4of4_share']<=GATE_MAX_EASY_SHARE)
    return rep

## 5. 파일럿 (카테고리별 40문항 × 4샘플) → 관문 판정 · 예상 시간 · 유닛
보고서에서 확인할 것:
1. **`gate_pass`:** 어려운 문항(`hard_share`) ≥ 15%, 4/4 비율 ≤ 70%인 카테고리만 통과. 통과하지 못한 카테고리는 학습 신호가 없으므로 `EXCLUDE_CATEGORIES`에 넣습니다.
2. **`zero_share`:** 4번 모두 틀린 문항 비율. 너무 높으면(약 50% 이상) 해당 카테고리는 4B에게 너무 어렵거나 정답 판정이 틀린 것입니다. `sample_accuracy`와 함께 보세요.
3. **`truncated_sample_rate`와 `tokens_median`:** 6,144토큰에서 잘리는 비율. 잘림이 많은 카테고리는 길이 제어 학습의 대상이지만 정답 풀이 수가 줄어듭니다.
4. **전체 생성 예상 시간·유닛:** 파일럿은 작은 배치라 보수적으로 느리게 잡힙니다.
`PILOT_ONLY=True`이면 여기서 멈춥니다. 파일럿 결과는 전체 실행에서 재사용됩니다.

In [ ]:
train_c=cands[cands.split=='train']
pilot=pd.concat([g.sample(n=min(PILOT_PER_CATEGORY,len(g)),random_state=SEED) for _,g in train_c.groupby('category')],ignore_index=True)
print('파일럿 문항:',len(pilot),pilot.category.value_counts().to_dict())
run_generation(pilot,'pilot')
gens=load_generations()
pq=build_quality(pilot,gens)
assert len(pq)==len(pilot),f'파일럿 생성 누락: {len(pilot)-len(pq)}'
prep=category_report(pq)
timing=[json.loads(l) for l in TIMING_FILE.open()] if TIMING_FILE.exists() else []
t=[x for x in timing if x['questions']>0]
rate=(sum(x['questions'] for x in t)/sum(x['generation_seconds'] for x in t)) if t else None
active=cands[~cands.category.isin(EXCLUDE_CATEGORIES)]
total_gen=min(len(active),MAX_GEN_QUESTIONS) if MAX_GEN_QUESTIONS else len(active)
remaining=total_gen-len(set(active.uid)&set(gens))
eta_h=(remaining/rate/3600) if rate else None
pilot_report={'category':prep,'questions_per_second':rate,'planned_generation_questions':total_gen,'remaining_questions':remaining,
              'estimated_remaining_hours':eta_h,'estimated_units_at_12_per_hour':(eta_h*12 if eta_h else None),
              'recommended_exclude':[c for c,v in prep.items() if not v['gate_pass']],
              'note':'파일럿은 작은 배치라 처리량이 낮게 잡힘(보수적). 모델 로딩 시간 제외.'}
(ROOT/'virl_pilot_report.json').write_text(jdump(pilot_report))
print(jdump(pilot_report))
tab=pd.DataFrame(prep).T[['questions','sample_accuracy','hard_share','easy_4of4_share','zero_share','truncated_sample_rate','tokens_median','gate_pass']]
print('\n',tab.to_string())
if PILOT_ONLY:
    raise RuntimeError('파일럿 완료: virl_pilot_report.json을 확인하세요. 통과하지 못한 카테고리를 EXCLUDE_CATEGORIES에 넣고, '
                       '필요하면 MAX_GEN_QUESTIONS를 조절한 뒤 PILOT_ONLY=False로 바꿔 설정 셀과 이 셀부터 다시 실행하세요 (생성 결과는 재사용됩니다).')

## 6. 전체 생성
이미 생성한 문항(파일럿 포함)은 건너뜁니다. `EXCLUDE_CATEGORIES`와 `MAX_GEN_QUESTIONS`를 반영합니다. 세션이 끊기면 설정 셀부터 다시 실행하면 이어서 생성합니다.

In [ ]:
gen_frame=cands[~cands.category.isin(EXCLUDE_CATEGORIES)]
if MAX_GEN_QUESTIONS and len(gen_frame)>MAX_GEN_QUESTIONS:
    frac=MAX_GEN_QUESTIONS/len(gen_frame)
    gen_frame=pd.concat([g.sample(n=max(1,int(round(len(g)*frac))),random_state=SEED) for _,g in gen_frame.groupby('category')],ignore_index=True)
print('생성 대상:',len(gen_frame),gen_frame.category.value_counts().to_dict())
run_generation(gen_frame,'full')
gens=load_generations()
missing=sorted(set(gen_frame.uid)-set(gens))
assert not missing,f'생성 누락 {len(missing)}문항. 이 셀을 다시 실행하면 이어서 생성합니다.'
print('생성 완료:',len(gen_frame),'문항 ×',RFT_SAMPLES)

## 7. 채점 · 선별 · 보고서
- **어려운 문항**(주관식 1~3/4, 객관식 2~3/4)의 정답 풀이(길이 중앙값에 가장 가까운 것)를 학습 타겟으로 씁니다.
- **앵커:** 4/4 정답 문항을 어려운 문항 대비 약 `ANCHOR_FRACTION/(1−ANCHOR_FRACTION)` 비율만 카테고리 비례로 추가합니다.
- **상한:** 카테고리 상한(`CATEGORY_CAP`)과 전체 상한(`MAX_TRAIN_SAMPLES`)을 적용합니다.
- 저장: `virl_selection_report.json`(집계, 공개 가능), `virl_selected_PRIVATE.jsonl`(원문, **GitHub 금지**).

In [ ]:
quality=build_quality(gen_frame,gens)
creport=category_report(quality)
def subsample(frame,n,seed_tag):
    if len(frame)<=n: return frame
    return frame.sample(n=n,random_state=int(hashlib.sha256(f'{SEED}|{seed_tag}'.encode()).hexdigest()[:8],16))
hard=quality[quality.status=='hard']; easy=quality[quality.status=='easy_4of4']
parts=[]
for cat,g in hard.groupby('category'):
    parts.append(subsample(g,CATEGORY_CAP.get(cat,10**9),f'cap|{cat}'))
hard_sel=pd.concat(parts,ignore_index=True) if parts else hard
hard_sel=subsample(hard_sel,int(MAX_TRAIN_SAMPLES*(1-ANCHOR_FRACTION)),'hard_total')
n_anchor=int(round(len(hard_sel)*ANCHOR_FRACTION/(1-ANCHOR_FRACTION)))
anch=[]
for cat,g in easy.groupby('category'):
    share=(hard_sel.category==cat).sum()/max(1,len(hard_sel))
    anch.append(subsample(g,int(round(n_anchor*share)),f'anchor|{cat}'))
anchors=pd.concat(anch,ignore_index=True) if anch else easy.head(0)
selected=pd.concat([hard_sel.assign(role='hard'),anchors.assign(role='anchor')],ignore_index=True)
sel_train=selected[selected.split=='train'].reset_index(drop=True); sel_val=selected[selected.split=='val'].reset_index(drop=True)
toks=sel_train.target_tokens
selection_report={'by_category':creport,'status_counts':{k:int(v) for k,v in quality.status.value_counts().items()},
    'selected_train':len(sel_train),'selected_val':len(sel_val),'roles':{k:int(v) for k,v in sel_train.role.value_counts().items()},
    'selected_train_by_source':{f'{c}/{k}':int(n) for (c,k),n in sel_train.groupby(['category','kind']).size().items()},
    'target_tokens':{'median':float(toks.median()),'p90':float(toks.quantile(0.9)),'max':int(toks.max())},
    'reference_baseline_mmmu_median_output_tokens':480,
    'rule':f'hard: open {HARD_RULES["open"]} mc {HARD_RULES["mc"]} of {RFT_SAMPLES}; anchors 4/4 ≤ {ANCHOR_FRACTION:.0%}; pick={TARGET_PICK}; {MIN_TARGET_TOKENS}-{MAX_TARGET_TOKENS} tokens'}
(ROOT/'virl_selection_report.json').write_text(jdump(selection_report))
with (ROOT/'virl_selected_PRIVATE.jsonl').open('w',encoding='utf-8') as f:
    for r in selected.drop(columns=['tokens_all','tokens_correct']).to_dict('records'): f.write(json.dumps(r,ensure_ascii=False,default=lambda x:x.item())+'\n')
print(jdump(selection_report))
assert len(sel_train)>=MIN_TRAIN_SAMPLES,f'학습 샘플이 너무 적습니다: {len(sel_train)} (< {MIN_TRAIN_SAMPLES}). 파일럿 관문과 EXCLUDE_CATEGORIES를 다시 검토하세요.'

## 8. SFT 데이터 구성 · collator 사전 검사
입력은 이미지 + 평가와 같은 프롬프트(`render_text`), 출력은 선별된 base 모델 응답 원문입니다. `images` 리스트 열을 씁니다(TRL 0.24 collator 요구사항). 학습 해상도는 평가와 같습니다.

In [ ]:
from datasets import Dataset, Image as HFImage
try:
    from datasets import List as HFList
except ImportError:
    from datasets import Sequence as HFList
from transformers import AutoProcessor

def to_sft(r):
    return {'images':[r['image']],
            'prompt':[{'role':'user','content':[{'type':'image'},{'type':'text','text':student_text(r)}]}],
            'completion':[{'role':'assistant','content':[{'type':'text','text':r['target']}]}]}
train_ds=Dataset.from_list([to_sft(r) for r in sel_train.to_dict('records')]).cast_column('images',HFList(HFImage()))
val_ds=Dataset.from_list([to_sft(r) for r in sel_val.to_dict('records')]).cast_column('images',HFList(HFImage())) if len(sel_val) else None
processor=AutoProcessor.from_pretrained(MODEL_ID,revision=revs['model'])
processor.image_processor.size={'shortest_edge':MIN_PIXELS,'longest_edge':MAX_PIXELS}   # 평가 해상도와 동일

from trl.trainer.sft_trainer import DataCollatorForVisionLanguageModeling
probe=DataCollatorForVisionLanguageModeling(processor=processor,completion_only_loss=True)([train_ds[i] for i in range(2)])
lab,att=probe['labels'],probe['attention_mask'].bool()
assert ((lab!=-100)&att).any(dim=1).all() and ((lab==-100)&att).any(dim=1).all() and (lab[~att]==-100).all()
assert probe['image_grid_thw'].shape[0]==2 and torch.isfinite(probe['pixel_values']).all()
print('Train',len(train_ds),'Val',0 if val_ds is None else len(val_ds))
print('collator 검사 통과: 입력',list(probe['input_ids'].shape),'| 학습 토큰',(lab!=-100).sum(dim=1).tolist(),
      '| image_grid_thw',probe['image_grid_thw'].tolist())
del probe;gc.collect()

## 9. BF16 LoRA 학습 · adapter 4개 저장
LoRA r=16, α=32, q/k/v/o(언어 모델만), 학습률 2e-5(이전 RFT의 1e-5보다 높음: 어려운 문항 위주라 신호가 있고 변화가 필요함), cosine, warmup 3%, **2 epoch**, gradient accumulation 8. 25/50/75/100% 지점에서 adapter를 저장하므로 **50%는 1 epoch 끝** 지점입니다. 세션이 끊기면 마지막 checkpoint부터 이어서 학습합니다.

In [ ]:
from transformers import Qwen3VLForConditionalGeneration, TrainerCallback
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

class SaveQuartileAdapters(TrainerCallback):
    def __init__(self,root,percentages=ADAPTER_SAVE_PCTS):
        self.root=Path(root);self.percentages=percentages;self.milestones={}
    def on_train_begin(self,args,state,control,**kw):
        total=int(state.max_steps); assert total>=4,total
        self.milestones={max(1,math.ceil(total*p/100)):p for p in self.percentages}
        assert len(self.milestones)==4,self.milestones
        print('Adapter 저장 step:',self.milestones)
    def on_step_end(self,args,state,control,**kw):
        pct=self.milestones.get(int(state.global_step))
        if pct is not None and state.is_world_process_zero:
            m=kw['model']; assert hasattr(m,'peft_config')
            folder=self.root/f'adapter_{pct:03d}pct'; folder.mkdir(parents=True,exist_ok=True)
            m.save_pretrained(str(folder),safe_serialization=True)
            (folder/'training_step.json').write_text(json.dumps({'target_pct':pct,'optimizer_step':int(state.global_step),
                'max_optimizer_steps':int(state.max_steps),'train_samples':len(train_ds),'seed':SEED,
                'model_revision':revs['model']},indent=2))
            print(f'[LoRA Adapter] {pct}% 저장 (step {state.global_step}) -> {folder}')
        return control

run_dir=ROOT/RUN_NAME; run_dir.mkdir(parents=True,exist_ok=True)
run_manifest={'train_uid_sha256':hashlib.sha256('\n'.join(sel_train.uid).encode()).hexdigest(),'train_count':len(sel_train),
              'target_sha256':hashlib.sha256('\n'.join(sel_train.target).encode()).hexdigest(),
              'seed':SEED,'model_revision':revs['model'],'lr':LEARNING_RATE,'epochs':NUM_TRAIN_EPOCHS,
              'grad_accum':GRADIENT_ACCUMULATION,'min_pixels':MIN_PIXELS,'max_pixels':MAX_PIXELS,
              'rft':{'samples':RFT_SAMPLES,'min_correct':RFT_MIN_CORRECT,'max_tokens':GEN_MAX_TOKENS,'pick':TARGET_PICK,
                     'target_tokens':[MIN_TARGET_TOKENS,MAX_TARGET_TOKENS]},'data_signature':DATA_SIG}
mf=run_dir/'run_manifest.json'
if mf.exists(): assert json.loads(mf.read_text())==run_manifest,'이전 checkpoint와 데이터/설정이 다릅니다. RUN_NAME을 바꾸세요.'
else: mf.write_text(json.dumps(run_manifest,indent=2))

model=Qwen3VLForConditionalGeneration.from_pretrained(MODEL_ID,revision=revs['model'],dtype=torch.bfloat16,
                                                      attn_implementation='sdpa',device_map='auto')
model.config.use_cache=False
lora=LoraConfig(r=16,lora_alpha=32,lora_dropout=0.05,bias='none',target_modules=['q_proj','k_proj','v_proj','o_proj'])
args=SFTConfig(output_dir=str(run_dir),seed=SEED,data_seed=SEED,num_train_epochs=NUM_TRAIN_EPOCHS,max_steps=-1,
    per_device_train_batch_size=1,per_device_eval_batch_size=1,gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,lr_scheduler_type='cosine',warmup_ratio=0.03,bf16=True,
    gradient_checkpointing=True,gradient_checkpointing_kwargs={'use_reentrant':False},
    max_length=None,completion_only_loss=True,logging_steps=25,save_strategy='steps',save_steps=SAVE_STEPS,
    save_total_limit=1,eval_strategy='no',optim='adamw_torch',report_to='none',dataloader_num_workers=0,
    remove_unused_columns=False,packing=False,max_grad_norm=1.0)
trainer=SFTTrainer(model=model,args=args,train_dataset=train_ds,eval_dataset=val_ds,peft_config=lora,
                   processing_class=processor,callbacks=[SaveQuartileAdapters(run_dir/'adapters')])
trainer.model.print_trainable_parameters()
names=[n for n,_ in trainer.model.named_parameters() if 'lora_' in n]
assert names and not any('.visual.' in n for n in names),'LoRA가 비전 타워에 붙었습니다'
b=trainer.data_collator([trainer.train_dataset[0]])
print('첫 샘플 학습 토큰:',int((b['labels']!=-100).sum()),'| 전체 입력',b['input_ids'].shape[1]); del b
ckpt=get_last_checkpoint(str(run_dir))
print('Resume checkpoint:',ckpt)
train_result=trainer.train(resume_from_checkpoint=ckpt)
print('Train metrics:',train_result.metrics)

In [ ]:
paths={p:run_dir/'adapters'/f'adapter_{p:03d}pct' for p in ADAPTER_SAVE_PCTS}
for p,folder in paths.items():
    assert (folder/'adapter_model.safetensors').is_file(),f'{p}% adapter 없음: {folder}'
val_metrics=trainer.evaluate() if val_ds is not None else {}
(run_dir/'val_loss.json').write_text(jdump(val_metrics))
(run_dir/'training_configuration.json').write_text(jdump({
    'method':f'RFT on hard questions (open 1-3/4, mc 2-3/4 correct; 4/4 anchors {ANCHOR_FRACTION:.0%}): base Qwen3-VL-4B samples, {TARGET_PICK} correct sample',
    'student_model':MODEL_ID,'revisions':revs,'train_rows':len(train_ds),'val_rows':0 if val_ds is None else len(val_ds),
    'train_by_source':selection_report['selected_train_by_source'],'lr':LEARNING_RATE,'epochs':NUM_TRAIN_EPOCHS,
    'lora':{'r':16,'alpha':32,'dropout':0.05,'target_modules':['q_proj','k_proj','v_proj','o_proj']},
    'min_pixels':MIN_PIXELS,'max_pixels':MAX_PIXELS,'prompt':'mmmu_eval.prompt.render_text (final_16k)',
    'generation':{k:EVAL_CFG[k] for k in ('temperature','top_p','top_k','presence_penalty','repetition_penalty','seed')}|
                 {'samples':RFT_SAMPLES,'max_tokens':GEN_MAX_TOKENS,'engine':'vllm==0.11.0'},
    'data':'ViRL39K slice (English, single image, leakage-screened vs MMMU val/test/dev + MMMU-Pro); open answers verified with boxed/Answer extraction + numeric tolerance + math-verify',
    'adapter_paths':{str(p):str(f) for p,f in paths.items()},'versions':versions,
    'metrics':{k:float(v) for k,v in train_result.metrics.items() if isinstance(v,(int,float))},
    'val_metrics':{k:float(v) for k,v in val_metrics.items() if isinstance(v,(int,float))}}))
print('Adapters:',{p:str(f) for p,f in paths.items()})
print('Val:',val_metrics)

## 10. 평가용 병합
평가 코드는 병합된 모델만 읽습니다. 기본은 100%(2 epoch 끝)이고, 학습 로그가 과하다고 느껴지면 `MERGE_ADAPTER_PCT=50`(1 epoch 끝)으로 바꿔 이 셀만 다시 실행하세요. 평가는 `MMDL/virl39k/eval/run_virl39k_rft_mmmu_900_drive_only.ipynb`에서 같은 `ADAPTER_PCT`로 실행합니다.

In [ ]:
MERGE_ADAPTER_PCT=100
from peft import PeftModel
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
run_dir=ROOT/RUN_NAME
selected_adapter=run_dir/'adapters'/f'adapter_{MERGE_ADAPTER_PCT:03d}pct'
assert (selected_adapter/'adapter_model.safetensors').exists(),f'Adapter 없음: {selected_adapter}'
merged_dir=run_dir/f'merged_model_{MERGE_ADAPTER_PCT:03d}pct_for_eval'
if (merged_dir/'config.json').exists():
    print('이미 병합됨:',merged_dir)
else:
    for n in ('trainer','model'): globals().pop(n,None)
    gc.collect();torch.cuda.empty_cache()
    base=Qwen3VLForConditionalGeneration.from_pretrained(MODEL_ID,revision=revs['model'],dtype=torch.bfloat16,
                                                         attn_implementation='sdpa',device_map='auto')
    merged=PeftModel.from_pretrained(base,str(selected_adapter)).merge_and_unload()
    tmp=merged_dir.with_name(merged_dir.name+'.tmp'); shutil.rmtree(tmp,ignore_errors=True)
    merged.save_pretrained(str(tmp),safe_serialization=True,max_shard_size='4GB')
    AutoProcessor.from_pretrained(MODEL_ID,revision=revs['model']).save_pretrained(str(tmp))   # 평가용 processor는 원본 그대로
    (tmp/'merge_info.json').write_text(json.dumps({'base':MODEL_ID,'base_revision':revs['model'],
        'adapter':str(selected_adapter),'adapter_pct':MERGE_ADAPTER_PCT,'method':'virl39k_rft4b'},indent=2))
    tmp.rename(merged_dir)
    del merged,base;gc.collect();torch.cuda.empty_cache()
print('병합 모델:',merged_dir)

In [ ]:
print('모든 셀 완료.')
if AUTO_SHUTDOWN:
    from google.colab import runtime
    print('60초 뒤 런타임을 종료합니다. 중단하려면 셀을 정지하세요.')
    time.sleep(60); runtime.unassign()